# 03 — Agent 1 : analyse multi-échelles de la géométrie

Ce notebook analyse uniquement les caractéristiques déductibles du GPX.

Caractéristiques V2 :

```text
PENTE — direction
- montee
- descente

PENTE — intensité
- pente_douce
- forte_pente

GÉOMÉTRIE
- lacets
```

`sinuosite` est volontairement supprimée de la V2 : elle apporte trop peu
par rapport à `lacets` pour le démonstrateur actuel.

Le LLM choisit une ou plusieurs fenêtres autorisées.
Les calculs sont ensuite faits par Python.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import requests

## 0. Charger la route normalisée du notebook 02

In [ ]:
route_path = Path("data/output/route_resampled.csv")

if not route_path.exists():
    raise FileNotFoundError(
        "Exécute d'abord le notebook 02 pour créer "
        "data/output/route_resampled.csv."
    )

route = pd.read_csv(route_path)

required_columns = {"distance_m", "lat", "lon", "elev_m"}
missing = required_columns - set(route.columns)

if missing:
    raise ValueError(f"Colonnes manquantes : {sorted(missing)}")

print("Route chargée :", route_path)
print("Points :", len(route))
print("Longueur :", round(route["distance_m"].iloc[-1], 1), "m")
display(route.head())

## 1. Configuration sémantique

On sépare explicitement deux dimensions incompatibles en leur sein :

```text
slope_direction :
    montee XOR descente

slope_intensity :
    pente_douce XOR forte_pente
```

Le notebook 03 conserve encore toutes les observations.
C'est le notebook 04 qui appliquera les contraintes `XOR` sur les portions finales.

In [ ]:
FEATURE_CONFIG = {
    "forte_pente": {
        "family": "slope_intensity",
        "detector": "slope",
        "allowed_windows_m": [10, 25, 50],
    },
    "pente_douce": {
        "family": "slope_intensity",
        "detector": "slope",
        "allowed_windows_m": [25, 50, 100],
    },
    "montee": {
        "family": "slope_direction",
        "detector": "slope",
        "allowed_windows_m": [50, 100, 200],
    },
    "descente": {
        "family": "slope_direction",
        "detector": "slope",
        "allowed_windows_m": [50, 100, 200],
    },
    "lacets": {
        "family": "geometry",
        "detector": "switchbacks",
        "allowed_windows_m": [75, 150, 250],
    },
}

## 2. Détecteur de pente

Important :

- `montee` / `descente` dépendent du **signe** de la pente ;
- `pente_douce` / `forte_pente` dépendent de sa **valeur absolue**.

Ainsi une descente peut être douce ou forte, et une montée aussi.

In [ ]:
def compute_window_slopes(route, window_m=25, step_m=10):
    distances = route["distance_m"].to_numpy(dtype=float)
    elev = route["elev_m"].to_numpy(dtype=float)

    rows = []
    start = 0.0
    end_total = float(distances[-1])

    while start + window_m <= end_total:
        end = start + window_m

        z0 = np.interp(start, distances, elev)
        z1 = np.interp(end, distances, elev)

        slope_pct = 100.0 * (z1 - z0) / window_m

        rows.append({
            "start_m": float(start),
            "end_m": float(end),
            "slope_pct": float(slope_pct),
        })

        start += step_m

    return pd.DataFrame(rows)


def detect_slope_feature(route, feature, window_m):
    slopes = compute_window_slopes(
        route,
        window_m=window_m,
        step_m=10,
    )

    abs_slope = slopes["slope_pct"].abs()

    if feature == "forte_pente":
        mask = abs_slope >= 12.0

    elif feature == "pente_douce":
        mask = (abs_slope >= 3.0) & (abs_slope < 8.0)

    elif feature == "montee":
        mask = slopes["slope_pct"] >= 5.0

    elif feature == "descente":
        mask = slopes["slope_pct"] <= -5.0

    else:
        raise ValueError(f"Feature de pente inconnue : {feature}")

    out = slopes.loc[mask].copy()
    out["feature"] = feature
    out["family"] = FEATURE_CONFIG[feature]["family"]
    out["resolution_m"] = float(window_m)

    # Confiance V1/V2 : score simple basé sur l'amplitude de la pente.
    out["confidence"] = np.minimum(
        1.0,
        np.maximum(0.0, abs(out["slope_pct"]) / 20.0)
    )

    return out

## 3. Détecteur de lacets

Détection géométrique indépendante de la pente.

In [ ]:
def bearing_deg(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dlambda = math.radians(lon2 - lon1)

    y = math.sin(dlambda) * math.cos(phi2)
    x = (
        math.cos(phi1) * math.sin(phi2)
        - math.sin(phi1) * math.cos(phi2) * math.cos(dlambda)
    )

    return (math.degrees(math.atan2(y, x)) + 360.0) % 360.0


def angular_difference(a, b):
    return (b - a + 180.0) % 360.0 - 180.0


def detect_switchbacks(
    route,
    window_m=150,
    step_m=25,
    min_turn_deg=35,
    min_turns=3,
):
    d = route["distance_m"].to_numpy(dtype=float)
    lat = route["lat"].to_numpy(dtype=float)
    lon = route["lon"].to_numpy(dtype=float)

    bearings = np.array([
        bearing_deg(lat[i], lon[i], lat[i + 1], lon[i + 1])
        for i in range(len(route) - 1)
    ])

    turns = np.array([
        angular_difference(bearings[i], bearings[i + 1])
        for i in range(len(bearings) - 1)
    ])

    turn_distance = d[1:-1]
    results = []

    start = 0.0

    while start + window_m <= d[-1]:
        end = start + window_m
        idx = (turn_distance >= start) & (turn_distance <= end)

        local_turns = turns[idx]
        significant = local_turns[np.abs(local_turns) >= min_turn_deg]

        signs = np.sign(significant)
        alternations = (
            int(np.sum(signs[1:] * signs[:-1] < 0))
            if len(signs) >= 2
            else 0
        )

        if len(significant) >= min_turns and alternations >= 2:
            score = min(
                1.0,
                0.2 * len(significant) + 0.15 * alternations
            )

            results.append({
                "feature": "lacets",
                "family": "geometry",
                "start_m": float(start),
                "end_m": float(end),
                "resolution_m": float(window_m),
                "confidence": float(score),
                "direction_changes": int(len(significant)),
                "alternations": alternations,
            })

        start += step_m

    return pd.DataFrame(results)

## 4. Client Ollama

Le LLM choisit uniquement les fenêtres.
Il ne calcule aucune pente et aucun lacet.

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.1:8b"


def ask_ollama_json(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "stream": False,
            "format": "json",
            "options": {"temperature": 0},
        },
        timeout=120,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Ollama HTTP {response.status_code}: {response.text}"
        )

    content = response.json()["message"]["content"]
    return json.loads(content)

## 5. Agent 1 — choix multi-échelles

In [ ]:
def choose_windows(feature, route, use_llm=True):
        cfg = FEATURE_CONFIG[feature]
        allowed = cfg["allowed_windows_m"]

        if not use_llm:
            return allowed

        prompt = f'''
Tu es un agent d'orchestration géospatiale.

Caractéristique : {feature}
Famille sémantique : {cfg["family"]}
Détecteur : {cfg["detector"]}
Fenêtres autorisées : {allowed}
Longueur de l'itinéraire : {route["distance_m"].iloc[-1]:.1f} m
Résolution spatiale de base : 10 m

Tu ne calcules aucune métrique géométrique.
Tu peux choisir UNE OU PLUSIEURS fenêtres parmi la liste autorisée.
Ne retourne jamais une valeur absente de la liste.

Réponds uniquement avec un JSON :
{{"windows_to_test_m": [25, 50]}}
'''

        try:
            data = ask_ollama_json(prompt)

            chosen = [
                int(w)
                for w in data["windows_to_test_m"]
                if int(w) in allowed
            ]

            if not chosen:
                raise ValueError("Aucune fenêtre valide retournée.")

            print(
                f"{feature} — réponse Agent 1 :",
                {"windows_to_test_m": chosen}
            )

            return sorted(set(chosen))

        except Exception as exc:
            print(
                f"{feature} — LLM indisponible/réponse invalide :",
                exc
            )
            print("Fallback : toutes les fenêtres autorisées.")
            return allowed

## 6. Exécuter les caractéristiques géométriques

In [ ]:
def run_feature_detection(route, feature, use_llm=True):
    windows = choose_windows(
        feature,
        route,
        use_llm=use_llm,
    )

    detector = FEATURE_CONFIG[feature]["detector"]
    frames = []

    for window in windows:
        if detector == "slope":
            df = detect_slope_feature(
                route,
                feature,
                window,
            )

        elif detector == "switchbacks":
            df = detect_switchbacks(
                route,
                window_m=window,
            )

        else:
            raise ValueError(
                f"Détecteur inconnu : {detector}"
            )

        if not df.empty:
            frames.append(df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

## 7. Test complet de l'Agent 1

Pour utiliser Ollama :

```python
USE_LLM = True
```

Pour tester toutes les fenêtres sans LLM :

```python
USE_LLM = False
```

In [ ]:
USE_LLM = True

FEATURES_TO_TEST = [
    "forte_pente",
    "pente_douce",
    "montee",
    "descente",
    "lacets",
]

detection_results = {}

for feature in FEATURES_TO_TEST:
    print("\n" + "=" * 70)
    print("CARACTÉRISTIQUE :", feature)

    result = run_feature_detection(
        route,
        feature,
        use_llm=USE_LLM,
    )

    detection_results[feature] = result

    print("Nombre de détections :", len(result))

    if not result.empty:
        display(result.head(10))
    else:
        print("Aucune détection.")

## 8. Résumé

In [ ]:
summary = pd.DataFrame([
    {
        "feature": feature,
        "family": FEATURE_CONFIG[feature]["family"],
        "n_detections": len(df),
        "resolutions_m": (
            sorted(df["resolution_m"].unique().tolist())
            if not df.empty
            else []
        ),
    }
    for feature, df in detection_results.items()
])

display(summary)

## 9. Export des annotations atomiques

Le notebook 04 utilisera toutes les preuves multi-échelles.
Il résoudra ensuite les incompatibilités sémantiques.

In [ ]:
STANDARD_COLUMNS = {
    "feature",
    "family",
    "start_m",
    "end_m",
    "resolution_m",
    "confidence",
}

atomic_annotations = []

for feature, df in detection_results.items():
    if df.empty:
        continue

    for _, row in df.iterrows():
        metrics = {}

        for column in df.columns:
            if column in STANDARD_COLUMNS:
                continue

            value = row[column]

            if pd.isna(value):
                continue

            if hasattr(value, "item"):
                value = value.item()

            metrics[column] = value

        atomic_annotations.append({
            "feature": str(row["feature"]),
            "family": str(
                row.get(
                    "family",
                    FEATURE_CONFIG[feature]["family"]
                )
            ),
            "start_m": float(row["start_m"]),
            "end_m": float(row["end_m"]),
            "resolution_m": float(row["resolution_m"]),
            "confidence": float(row["confidence"]),
            "metrics": metrics,
        })

output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

json_path = output_dir / "agent1_annotations_atomic.json"
csv_path = output_dir / "agent1_annotations_atomic.csv"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(
        atomic_annotations,
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame([
    {
        **{k: v for k, v in a.items() if k != "metrics"},
        "metrics_json": json.dumps(
            a["metrics"],
            ensure_ascii=False
        ),
    }
    for a in atomic_annotations
]).to_csv(csv_path, index=False)

print("Export géométrique terminé :")
print("-", json_path)
print("-", csv_path)
print("Annotations :", len(atomic_annotations))